#Trabalho T1
Grupo:
- Brenda
- Enzo Claudino
- Thainá
- Natália

Segue o notebook =)

Esta etapa inicial são só os imports chave e o carregamento dos arquivos fundamentais.

In [ ]:
#+++++++++++++++++++++++++++++++++++++++++++++++++++++
# Imports chave e montar o drive como local de arquivo
#+++++++++++++++++++++++++++++++++++++++++++++++++++++

!pip install FPDF
from google.colab import drive
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from sklearn.model_selection import LeaveOneOut, cross_val_score
import seaborn as sns
from IPython.display import display
from fpdf import FPDF
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

#+++++++++++++++++++++++++++++++++++++++++++++++++++++
# Buscar arquivo de dados no drive
#+++++++++++++++++++++++++++++++++++++++++++++++++++++

# Monta o Google Drive
drive.mount('/content/drive', force_remount=True)
# Caminho do Drive
caminho_drive = "/content/drive/MyDrive/analytics III/T1/franquia_lojas.csv"
df=pd.read_csv(caminho_drive)
#Caso você esteja rodando localmetne para facilitar o trabalho basta comentar o código anterior e descomentar o seguinte
#caminho_deive = "/content/franquia_lojas.csv"


#+++++++++++++++++++++++++++++++++++++++++++++++++++++
# Teste de sanidade do DF
#+++++++++++++++++++++++++++++++++++++++++++++++++++++

print(df.shape)
print(df.head)

#Parte 1: Limpeza dos dados e conhecimento da base:

#Limpeza de dados:

Para esta etapa desenvolvemos (no espírito caótico movido pela falta de concerta) o limpador de dados radical, uma função que trata dos dados em algumas etapas para que ele seja útil para o trabalho. A sequência que segue explica o processo de tratamento de dados aplicado, para auditoria convido a ver a função no ipynb.

1.	Remover duplicatas;
2.	Definir as colunas para avaliação de outliers e remover outliers pelo método de IQR;
3.	Remover as linhas com dados faltantes;
4.	Transformar as variáveis categóricas em dummies e garantir que os dados sejam úteis para modelagem;

A utilidade de cada uma das variáveis ainda não foi medida nesta etapa do projeto.

In [ ]:
# Limpeza de dados aprimorada

def limpador_de_dados_radical(df):
    print(f'Shape inicial: {df.shape}')

    # 1. Remover duplicatas
    df = df.drop_duplicates()

    # 2. Definir colunas numéricas REAIS para outliers (antes de criar dummies)
    # Lista explícita para evitar processar dummies de cidades/regiões como outliers
    cols_para_outliers = [
        'area_m2', 'populacao_1km', 'renda_media_domiciliar', 'fluxo_pedestres_dia',
        'vagas_estacionamento', 'distancia_concorrente_m', 'meses_operacao',
        'investimento_marketing_mensal', 'n_funcionarios', 'ticket_medio_rs',
        'n_transacoes_mes', 'nota_satisfacao_cliente', 'faturamento_medio_mensal'
    ]
    # Garantir que apenas colunas existentes no DF sejam usadas
    cols_para_outliers = [c for c in cols_para_outliers if c in df.columns]

    # 3. Remover Outliers (IQR)
    print(f'Removendo outliers das colunas principais: {cols_para_outliers}')
    for col in cols_para_outliers:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        inf = Q1 - 1.5 * IQR
        sup = Q3 + 1.5 * IQR
        # Filtra mantendo nulos para não dropar antes da hora
        df = df[(df[col] >= inf) & (df[col] <= sup) | (df[col].isna())]

    # 4. Remover linhas com valores faltantes (NaN)
    linhas_antes_nan = df.shape[0]
    df = df.dropna()
    print(f'Linhas removidas por conterem valores nulos: {linhas_antes_nan - df.shape[0]}')

    # 5. Transformar variáveis categóricas em dummies
    cols_cat = df.select_dtypes(include=['object']).columns.tolist()
    if cols_cat:
        df = pd.get_dummies(df, columns=cols_cat, drop_first=True)

    # 6. Converter booleanos para int (0 e 1)
    cols_bool = df.select_dtypes(include=['bool']).columns
    df[cols_bool] = df[cols_bool].astype(int)

    print(f'Novo shape após limpeza completa: {df.shape}')
    return df

# Recarregar o DF original para limpar do zero corretamente
df = pd.read_csv(caminho_drive)
df = limpador_de_dados_radical(df)
display(df.head())

#Exploração dos dados e conhecimento da base

Para esta etapa desenvolvemos a EDA Master Blaster, ela se propõe a demonstrar as características do DF por meio de gráficos e de tabelas explicativas dos dados de que estamos tratando.

Vale ressaltar que tanto este código quanto o anterior são reutilizados de projetos anteriores, portanto a organização não está 100% adequada e há alguns bugs.

---

Por favor não tente exportar o pdf da EDA pelo colab, ele vai travar por completo.
---

In [ ]:
# EDA Tbm como objeto pq sim

def EDA_Master_Blaster(df, export_pdf):
    export_pdf = export_pdf
    print("--- Informações Gerais ---")
    print(df.info())

    print("\n--- Valores Faltantes ---")
    print(df.isnull().sum())

    print("\n--- Estatísticas Descritivas ---")
    desc = df.describe()
    display(desc)

    print("\n--- Erro Padrão da Média (SEM) ---")
    sem_values = df.select_dtypes(include=[np.number]).sem()
    sem_df = pd.DataFrame(sem_values, columns=['Erro Padrão'])
    display(sem_df)

    print("\n--- RMSE ---")
    rmse_values = np.sqrt(np.mean((df.select_dtypes(include=[np.number]) - df.select_dtypes(include=[np.number]).mean())**2, axis=0))
    rmse_df = pd.DataFrame(rmse_values, columns=['RMSE'])
    display(rmse_df)

    print("\n--- Histogramas das Variáveis Numéricas ---")
    df.hist(figsize=(15, 10), bins=20, color='skyblue', edgecolor='black')
    plt.tight_layout()
    plt.savefig('eda_hist.png')
    plt.show()

    print("\n--- Identificação de Outliers (Boxplots Individuais) ---")
    features_num = df.select_dtypes(include=[np.number]).columns.tolist()
    n_features = len(features_num)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols

    fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(18, 4 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(features_num):
        sns.boxplot(data=df, y=col, ax=axes[i], color='lightcoral')
        axes[i].set_title(f'Boxplot de {col}')

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.savefig('eda_boxplots.png')
    plt.show()

    print("\n--- Matriz de Correlação (Heatmap) ---")
    plt.figure(figsize=(12, 8))
    corr = df.corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
    plt.title("Mapa de Calor de Correlação")
    plt.savefig('eda_corr.png')
    plt.show()

    print("\n--- Top 5 Variáveis mais Correlacionadas com Faturamento ---")
    target_corr = corr['faturamento_medio_mensal'].abs().sort_values(ascending=False)
    print(target_corr.head(6))

    print("\n--- Análise de Dispersão vs Faturamento ---")
    target = 'faturamento_medio_mensal'
    features = [col for col in df.columns if col != target]
    n_features_disp = len(features)
    n_rows_disp = (n_features_disp + n_cols - 1) // n_cols

    fig_disp, axes_disp = plt.subplots(nrows=n_rows_disp, ncols=n_cols, figsize=(18, 4 * n_rows_disp))
    axes_disp = axes_disp.flatten()

    for i, col in enumerate(features):
        sns.scatterplot(data=df, x=col, y=target, ax=axes_disp[i])
        axes_disp[i].set_title(f'{col} vs Faturamento')

    for j in range(i + 1, len(axes_disp)):
        fig_disp.delaxes(axes_disp[j])

    plt.tight_layout()
    plt.savefig('eda_dispersion.png')
    plt.show()

    print("\n--- Distribuição de Faturamento ---")
    plt.figure(figsize=(10, 6))
    sns.histplot(df['faturamento_medio_mensal'], bins=20, kde=True, color='skyblue')
    plt.title("Distribuição de Faturamento")
    plt.savefig('eda_target_dist.png')
    plt.show()

    print("\n--- Identificação de Outliers no Faturamento (Tabela) ---")
    Q1 = df['faturamento_medio_mensal'].quantile(0.25)
    Q3 = df['faturamento_medio_mensal'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df['faturamento_medio_mensal'] < lower_bound) | (df['faturamento_medio_mensal'] > upper_bound)]

    if not outliers.empty:
        display(outliers)
    else:
        print("Nenhum outlier detectado para a variável faturamento_medio_mensal.")

    if export_pdf == True:
        pdf = FPDF()
        pdf.set_auto_page_break(auto=True, margin=15)
        pdf.add_page()
        pdf.set_font("Arial", "B", 16)
        pdf.cell(200, 10, "Relatorio EDA Master Blaster", ln=True, align="C")
        pdf.ln(10)

        pdf.set_font("Arial", "B", 12)
        pdf.cell(0, 10, "1. Histogramas Gerais", ln=True)
        pdf.image('eda_hist.png', x=10, w=180)

        pdf.add_page()
        pdf.cell(0, 10, "2. Boxplots de Outliers", ln=True)
        pdf.image('eda_boxplots.png', x=10, w=180)

        pdf.add_page()
        pdf.cell(0, 10, "3. Matriz de Correlação", ln=True)
        pdf.image('eda_corr.png', x=10, w=160)

        pdf.add_page()
        pdf.cell(0, 10, "4. Dispersão vs Faturamento", ln=True)
        pdf.image('eda_dispersion.png', x=10, w=180)

        pdf.ln(5)
        pdf.cell(0, 10, "5. Distribuição do Faturamento", ln=True)
        pdf.image('eda_target_dist.png', x=10, w=160)

        pdf.output("Relatorio_EDA.pdf")
        print("\nRelatório PDF 'Relatorio_EDA.pdf' gerado com sucesso!")

    else:
        print("\nRelatório EDA não exportado para PDF.")

    print("\nEDA Master Blaster concluída com sucesso, agora basta dominar o universo")

# Chamando a função,o export em pdf está quebrado, não colocar true se não vai quebrar o código
EDA_Master_Blaster(df, False)

#Profiler

Dentre as lojas que estão no dataset das existentes, há um padrão entre as que tem boa performance ou não, portanto aqui segue um perfilador de todos estes dados. =)



In [ ]:

def profiler_apocaliptico_de_lojas(df_input, export_pdf=True):
    # 1. Definindo os grupos
    df_profiler = df_input.copy()
    df_profiler['grupo_performance'] = np.where(df_profiler['faturamento_medio_mensal'] > 140, 'Alta Performance', 'Performance Padrão')

    cols_interesse = [
        'area_m2', 'populacao_1km', 'renda_media_domiciliar', 'fluxo_pedestres_dia',
        'vagas_estacionamento', 'meses_operacao', 'investimento_marketing_mensal',
        'n_funcionarios', 'ticket_medio_rs', 'n_transacoes_mes', 'nota_satisfacao_cliente'
    ]

    # 2. Cálculos
    perfil = df_profiler.groupby('grupo_performance')[cols_interesse].agg(['mean', 'median', 'std']).T
    grupo_alta = df_profiler[df_profiler['grupo_performance'] == 'Alta Performance']
    grupo_padrao = df_profiler[df_profiler['grupo_performance'] == 'Performance Padrão']

    p_values = {}
    for col in cols_interesse:
        t_stat, p_val = stats.ttest_ind(grupo_alta[col], grupo_padrao[col], equal_var=False)
        p_values[col] = round(p_val, 4)

    perfil_final = perfil.reset_index()
    perfil_final.columns = ['Variavel', 'Stat', 'Alta Performance', 'Performance Padrão']
    perfil_final['P-Value'] = perfil_final['Variavel'].map(p_values)

    # 3. Visualização (Corrigindo o erro de palette/hue)
    print("=== PROFILER DE PERFORMANCE DE LOJAS ===")
    display(perfil_final)

    top_vars = sorted(p_values, key=p_values.get)[:3]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, var in enumerate(top_vars):
        sns.barplot(data=df_profiler, x='grupo_performance', y=var, hue='grupo_performance', ax=axes[i], palette='viridis', legend=False)
        axes[i].set_title(f'Diferença em {var}\n(p-value: {p_values[var]})')

    plt.tight_layout()
    plt.savefig('graficos_profiler.png')
    plt.show()

    # 4. Exportação para PDF
    if export_pdf:
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial", "B", 16)
        pdf.cell(200, 10, "Relatorio de Profiling de Lojas", ln=True, align="C")
        pdf.set_font("Arial", "", 12)
        pdf.ln(10)
        pdf.cell(200, 10, f"Lojas Alta Performance: {len(grupo_alta)}", ln=True)
        pdf.cell(200, 10, f"Lojas Performance Padrao: {len(grupo_padrao)}", ln=True)
        pdf.ln(5)

        # Tabela simplificada no PDF
        pdf.set_font("Arial", "B", 10)
        pdf.cell(40, 10, "Variavel", 1)
        pdf.cell(30, 10, "Media (Alta)", 1)
        pdf.cell(30, 10, "Media (Padrao)", 1)
        pdf.cell(30, 10, "P-Value", 1)
        pdf.ln()

        pdf.set_font("Arial", "", 9)
        for col in cols_interesse:
            m_alta = round(grupo_alta[col].mean(), 2)
            m_pad = round(grupo_padrao[col].mean(), 2)
            pdf.cell(40, 10, col, 1)
            pdf.cell(30, 10, str(m_alta), 1)
            pdf.cell(30, 10, str(m_pad), 1)
            pdf.cell(30, 10, str(p_values[col]), 1)
            pdf.ln()

        pdf.ln(10)
        pdf.image('graficos_profiler.png', x=10, w=190)
        pdf.output("Relatorio_Lojas.pdf")
        print("\nRelatório PDF gerado com sucesso: Relatorio_Lojas.pdf")

profiler_apocaliptico_de_lojas(df)

#Parte 2: Modelos exploratórioa com todas as variáveis:

##Regressão por Boostraping

O primeiro método que aplicamos nos nossos testes preliminares foi o bootstraping em 450 tentativas com algumas regras, as variáveis (tanto em quantidade quanto em seleção) foram aleatorizadas, mas todos os modelos resultantes devem no mínimo cumprir duas regras, a divisão entre treino e teste (já que há a etapa de teste) e a relevância estatística de todas as variáveis contidas (pelo menos 95%). Uma vez concluída a etapa aleatória, os modelos são carregados na memória e são selecionados os 10 modelos com maior R^2 nos dados de treinamento e estes são testados contra os dados de teste, uma vez isso feito o modelo com o maior R^2 nos dados de teste é exportado para avaliação humana.

In [ ]:
# Façamos os modelos de regressão linear com refino estatístico por método boostrap

def fazedor_de_modelos_para_dominação_do_universo(df):
    target = 'faturamento_medio_mensal'

    # 1. Tratamento interno de NaNs para evitar o erro MissingDataError
    df_clean = df.copy()
    cols_numericas = df_clean.select_dtypes(include=[np.number]).columns
    for col in cols_numericas:
        if df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

    # 2. Converte booleanos para float para evitar erro no statsmodels
    df_numeric = df_clean.astype(float)
    features_pool = [col for col in df_numeric.columns if col != target]

    # Separação Treino e Teste (questão 3)
    X = df_numeric[features_pool]
    y = df_numeric[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    modelos_resultados = []
    n_tentativas = 450

    print(f"Iniciando a busca aleatória por {n_tentativas} modelos com filtro de P-value...")

    for i in range(n_tentativas):
        # Escolhe um número aleatório de variáveis e quais variáveis
        k = random.randint(1, len(features_pool))
        selected_features = random.sample(features_pool, k)

        # Passo de Refino: Remover variáveis P > 0.05
        while len(selected_features) > 0:
            X_temp = sm.add_constant(X_train[selected_features])
            model_temp = sm.OLS(y_train, X_temp).fit()
            # Pega o maior p-value ignorando a constante
            p_values = model_temp.pvalues.drop('const', errors='ignore')
            max_p = p_values.max()

            if max_p > 0.05:
                excluded_feature = p_values.idxmax()
                selected_features.remove(excluded_feature)
            else:
                break

        if len(selected_features) > 0:
            # Treina modelo final da rodada com as selecionadas/refinadas
            lr = LinearRegression()
            lr.fit(X_train[selected_features], y_train)
            r2 = lr.score(X_train[selected_features], y_train)

            modelos_resultados.append({
                'features': selected_features,
                'r2_treino': r2
            })

    if not modelos_resultados:
        return "Nenhum modelo com variáveis significativas encontrado."

    # Ordena e pega os 10 melhores R2 de treino
    top_10_r2 = sorted(modelos_resultados, key=lambda x: x['r2_treino'], reverse=True)[:10]
    validacao_final = []

    print(f"\nTestando os {len(top_10_r2)} melhores modelos refinados contra os dados de teste...")

    for item in top_10_r2:
        cols = item['features']
        lr_val = LinearRegression()
        lr_val.fit(X_train[cols], y_train)

        y_pred = lr_val.predict(X_test[cols])
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)

        item['mse_teste'] = mse
        item['rmse_teste'] = rmse
        validacao_final.append(item)

    # O melhor modelo é o com menor RMSE
    melhor_modelo_info = min(validacao_final, key=lambda x: x['rmse_teste'])

    # Relatório Detalhado do vencedor
    X_vencedor_train = sm.add_constant(X_train[melhor_modelo_info['features']])
    modelo_final = sm.OLS(y_train, X_vencedor_train).fit()

    print("="*60)
    print("MELHOR MODELO REFINADO ENCONTRADO")
    print("="*60)
    print(f"Variáveis finais (P <= 0.05): {melhor_modelo_info['features']}")
    print(f"R-quadrado no Treino: {melhor_modelo_info['r2_treino']:.4f}")
    print(f"MSE no Teste: R$ {melhor_modelo_info['mse_teste']:.2f}")
    print(f"RMSE no Teste: R$ {melhor_modelo_info['rmse_teste']:.2f}")
    print(f"R-quadrado no Teste: {modelo_final.rsquared:.4f}")
    print("-"*60)
    print(modelo_final.summary())

    return modelo_final

# Executando a função
modelo_vencedor = fazedor_de_modelos_para_dominação_do_universo(df)

##LOOCV
O segundo modelo que testamos foi um de loocv usando todas as variáveis sem aleatoriedade neste aspecto, este modelo não foi funcional, mas vale ressaltar que apresentou um RMSE extremamente baixo em comparação com o incialmente visto na EDA.

In [ ]:
#Teste de LOOCV por esporte =)

def modelo_loocv_para_o_fim_do_universo(df):
  # Definindo a variável alvo primeiro para evitar o UnboundLocalError
  target = 'faturamento_medio_mensal'
  features = [col for col in df.columns if col != target]

  X = df[features]
  y = df[target]

  # Inicializando o modelo e o método LOOCV
  modelo_final = LinearRegression()
  loo = LeaveOneOut()

  # 1. Validação Cruzada (LOOCV)
  scores = cross_val_score(modelo_final, X, y, cv=loo, scoring='neg_mean_squared_error')
  mse_scores = -scores
  mse_loocv = np.mean(mse_scores)
  rmse_loocv = np.sqrt(mse_loocv)

  # 2. Treinamento no dataset completo para extrair coeficientes
  modelo_final.fit(X, y)

  print("="*60)
  print("RESULTADOS DO MODELO FINAL E VALIDAÇÃO LOOCV")
  print("="*60)
  print(f"Variáveis utilizadas: {features}")
  print(f"MSE (LOOCV): R$ {mse_loocv:.4f}")
  print(f"RMSE (LOOCV): R$ {rmse_loocv:.4f}")
  print("-"*60)
  print("COEFICIENTES DO MODELO:")
  print(f"Intercepto (Bias): {modelo_final.intercept_:.4f}")

  coef_df = pd.DataFrame({'Variável': features, 'Coeficiente': modelo_final.coef_})
  print(coef_df.to_string(index=False))
  print("-"*60)
  print("O modelo está validado e os coeficientes acima representam a equação final.")

#Executando a Função
modelo_loocv = modelo_loocv_para_o_fim_do_universo(df)

##Stepwise (iteração passo-a-passo)

O terceiro modelo que fizemos é um modelo construído por foward stepwise, usando como critério apenas a relevância estatística de 95% para adição de qualquer variável no modelo (recalculado a cada iteração do modelo), o RMSE deste output foi de 3.917.

In [ ]:
target = 'faturamento_medio_mensal'
# Get all potential numerical predictors, excluding target and id_loja
all_predictors = df.select_dtypes(include=np.number).columns.tolist()
all_predictors.remove(target)
if 'id_loja' in all_predictors:
    all_predictors.remove('id_loja')

selected_features = []
remaining_features = all_predictors[:]
best_model_summary = None
p_value_threshold_in = 0.05 # Threshold to add a variable

print("Iniciando seleção de variáveis forward (stepwise):\n")

while True:
    potential_adds = []

    # Try adding each remaining feature one by one to the current set of selected features
    for feature_to_add in remaining_features:
        current_candidate_features = selected_features + [feature_to_add]

        # Construct formula
        if not current_candidate_features:
            formula_str = f"{target} ~ 1"
        else:
            formula_str = f"{target} ~ {' + '.join(current_candidate_features)}"

        try:
            model = smf.ols(formula=formula_str, data=df).fit()

            if feature_to_add in model.pvalues.index:
                p_val_new_feature = model.pvalues[feature_to_add]
                potential_adds.append((p_val_new_feature, feature_to_add, model))
        except Exception:
            pass

    potential_adds.sort(key=lambda x: x[0])

    if potential_adds and potential_adds[0][0] < p_value_threshold_in:
        best_p_value, best_feature_to_add, current_best_model = potential_adds[0]
        selected_features.append(best_feature_to_add)
        remaining_features.remove(best_feature_to_add)
        best_model_summary = current_best_model

        print(f"--- Passo: Adicionando '{best_feature_to_add}' ---")
        print(f"Variáveis no modelo: {selected_features}")
        print(f"P-valor da nova variável ({best_feature_to_add}): {best_p_value:.4f}")
        print(f"R-quadrado Ajustado: {current_best_model.rsquared_adj:.4f}\n")
    else:
        break

print("--- Seleção de Variáveis Concluída ---")
print(f"Variáveis finais selecionadas: {selected_features}")

if best_model_summary:
    # Calculando MSE e RMSE do modelo final
    mse_final = best_model_summary.mse_resid
    rmse_final = np.sqrt(mse_final)

    print("\n" + "="*60)
    print("MÉTRICAS DE ERRO EM REAIS (Final Stepwise Model)")
    print("="*60)
    print(f"MSE em Reais: R$ {mse_final:.4f}")
    print(f"RMSE em Reais: R$ {rmse_final:.4f}")
    print("="*60)

    print("\n### Sumário do Modelo Final:")
    display(best_model_summary.summary())
else:
    print("Nenhuma variável foi selecionada.")

##Modelo de Regressão com Regularizaçã LASSO

Para finalizar os modelos tentativos – ainda sem a exclusão completa das variáveis inadequadas – aplicamos novamente o método do primeiro modelo, mas desta vez com a regularização LASSO das variáveis, é importante ressaltar que a regularização acontece antes do modelo passar por qualquer rankeamento (garantindo um modelo final mais representativo dado que qualquer função só será testada contra os dados de teste uma vez finalizada e regularizada) é também importante ressaltar que o alfa do LASSO também foi aleatorizado entre 0,001; 0,01; 0,1; 0,5; e 1.

In [ ]:
def fazedor_de_modelos_para_dominação_do_universo_reg(df_input, alphas=[0.001, 0.01, 0.1, 0.5, 1.0]):
    target = 'faturamento_medio_mensal'

    # 1. Tratamento interno de NaNs
    df_clean = df_input.copy()
    cols_numericas = df_clean.select_dtypes(include=[np.number]).columns
    for col in cols_numericas:
        if df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

    # 2. Converte booleanos para float
    df_numeric = df_clean.select_dtypes(include=[np.number]).astype(float)
    features_pool = [col for col in df_numeric.columns if col != target]

    # Separação Treino e Teste
    X = df_numeric[features_pool]
    y = df_numeric[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Padronização (importante para LASSO)
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=features_pool, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features_pool, index=X_test.index)

    modelos_resultados = []
    n_tentativas = 450

    print(f"Iniciando busca bootstrap com refino de P-valor e regularização LASSO...")

    for i in range(n_tentativas):
        k = random.randint(1, len(features_pool))
        selected_features = random.sample(features_pool, k)

        # --- PASSO 1: Refino por P-value (Estatística Clássica) ---
        while len(selected_features) > 0:
            X_temp = sm.add_constant(X_train[selected_features])
            model_temp = sm.OLS(y_train, X_temp).fit()
            p_values = model_temp.pvalues.drop('const', errors='ignore')
            max_p = p_values.max()

            if max_p > 0.05:
                excluded_feature = p_values.idxmax()
                selected_features.remove(excluded_feature)
            else:
                break

        # --- PASSO 2: Regularização LASSO sobre as variáveis significativas ---
        if len(selected_features) > 0:
            for a in alphas:
                lasso = Lasso(alpha=a, random_state=42)
                lasso.fit(X_train_scaled[selected_features], y_train)
                r2_tr = lasso.score(X_train_scaled[selected_features], y_train)

                modelos_resultados.append({
                    'features': selected_features,
                    'alpha': a,
                    'r2_treino': r2_tr,
                    'lasso_model': lasso
                })

    if not modelos_resultados:
        print("ERRO: Nenhum modelo com variáveis significativas (P < 0.05) foi encontrado nas 450 tentativas.")
        return None

    # 3. Filtragem Top 10 R2 de Treino conforme solicitado
    top_10_r2 = sorted(modelos_resultados, key=lambda x: x.get('r2_treino', -1), reverse=True)[:10]
    validacao_final = []

    print(f"Testando os {len(top_10_r2)} melhores modelos de treino contra os dados de teste...")

    for item in top_10_r2:
        cols = item['features']
        model = item['lasso_model']

        y_pred = model.predict(X_test_scaled[cols])
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)

        item['mse_teste'] = mse
        item['rmse_teste'] = rmse
        validacao_final.append(item)

    # O melhor modelo é o com menor RMSE entre os top R2
    melhor_modelo_info = min(validacao_final, key=lambda x: x['rmse_teste'])

    print("="*60)
    print("MELHOR MODELO (P-VALOR + LASSO + VALIDACAO R2) ENCONTRADO")
    print("="*60)
    print(f"Variáveis Significativas: {melhor_modelo_info['features']}")
    print(f"Melhor Alpha LASSO: {melhor_modelo_info['alpha']}")
    print(f"RMSE Final no Teste: R$ {melhor_modelo_info['rmse_teste']:.2f}")
    print(f"R-quadrado no Treino (Lasso): {melhor_modelo_info['r2_treino']:.4f}")
    print("="*60)

    # Sumário OLS final para conferência de coeficientes e p-valores
    X_final_ols = sm.add_constant(X_train[melhor_modelo_info['features']])
    print(sm.OLS(y_train, X_final_ols).fit().summary())

    return melhor_modelo_info['lasso_model']

if 'df' in locals():
    modelo_vencedor = fazedor_de_modelos_para_dominação_do_universo_reg(df)
else:
    print("Dataframe 'df' não encontrado.")

#Parte 3: Veto das variáveis para uso nos modelos finais

Uma vez trabalhados os dados e observados os modelos (assim como os avisos automáticos gerados pelo Statsmodel), algumas variáveis se provaram inviáveis para o uso no modelo final, cada uma delas, e suas respectivas explicações, está a seguir:

	“n_transacoes_mes”, “n_transacoes_mes”:
	Estas variáveis têm dois problemas, por um lado são variáveis que estão embutidas no faturamento mensal – o faturamento é dado pela fórmula F_mensal=N_(de transações mês)* T_médio – enquanto por outro lado são variáveis que só podem ser descobertas após a abertura da loja, portanto não estão nos dados para tomada de decisão;
	“id_loja”:
	É apenas uma variável de identificação, não é relevante para o modelo;
	“meses_operacao”, “nota_satisfacao_cliente”:
	Estas variáveis somente podem ser descobertas após a operação da loja.

Uma vez excluídas estas vaiáveis do pool de modelagem, podemos partir para as etapas finais
